# 09 – Gold dim_customer (SCD Type 2)

Bu notebook `workspace.gold.dim_customer` tablosunu kurar: önce bronze snapshot'ından ilk yükleme, sonra landing batch'leri 1 → 3 SCD2 MERGE ile.

**Bu hücre:** Tabloyu `CREATE OR REPLACE` ile yeniden oluşturur ve bronze `client` tablosundan yükler. `batch_log`'daki `dim_customer` satırlarını siler, tekrar çalıştırılabilir.

**Çıktı:** `rows`, `current_rows` ve `clients` 5.369 olmalı; NULL sayıları 0. Client 435 için `district_id` 20 ve `is_current` true olmalı.

In [0]:
import os
import sys
import importlib

from pyspark.sql import functions as F

repo_root = os.path.dirname(os.getcwd())
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

import silver_transforms
import silver_merge
import gold_dim_customer
importlib.reload(silver_transforms)
importlib.reload(silver_merge)
importlib.reload(gold_dim_customer)

gold_dim_customer.load_initial(spark)

df_dim = spark.table(gold_dim_customer.GOLD_TABLE)

df_summary = df_dim.agg(
    F.count("*").alias("rows"),
    F.count_if(F.col("is_current")).alias("current_rows"),
    F.countDistinct("client_id").alias("clients"),
    F.count_if(F.col("birth_date").isNull()).alias("null_birth_date"),
    F.count_if(F.col("district_name").isNull()).alias("null_district_name"),
    F.count_if(F.col("gender") == "F").alias("female"),
    F.min("customer_sk").alias("min_sk"),
    F.max("customer_sk").alias("max_sk"),
)
display(df_summary)

display(df_dim.filter(F.col("client_id") == 435))

## SCD2 MERGE: batch 1 → 3

**Bu hücre:** Landing `client` batch'lerini sırayla SCD2 MERGE ile uygular. Değişen müşterinin açık kaydı kapanır, yeni versiyon eklenir. `batch_log`'da SUCCESS olan batch atlanır.

**Çıktı:** Her batch için SUCCESS. Log'da `rows_inserted` / `rows_updated` değerleri batch 1: 13 / 8, batch 2: 24 / 19, batch 3: 9 / 5 olmalı. Özette `rows` 5.415, `current_rows` 5.383 (silver client ile aynı), `closed_rows` 32 olmalı.

In [0]:
for batch_id in [1, 2, 3]:
    result = gold_dim_customer.run_batch(spark, batch_id)
    print(f"batch {batch_id}: {result}")

df_log = spark.table("workspace.silver.batch_log").filter(F.col("table_name") == "dim_customer")
display(df_log.orderBy("batch_id"))

df_dim = spark.table(gold_dim_customer.GOLD_TABLE)
df_summary = df_dim.agg(
    F.count("*").alias("rows"),
    F.count_if(F.col("is_current")).alias("current_rows"),
    F.count_if(~F.col("is_current")).alias("closed_rows"),
)
display(df_summary)